<a href="https://colab.research.google.com/github/Monia1801/Prompt-Engineering/blob/main/Exp_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-text-splitters langchain-community pypdf faiss-cpu langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 28.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 71.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 589.8/589.8 kB 26.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.6 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstal

In [2]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from google.colab import userdata

/tmp/ipykernel_452/300875391.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
embeddings_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0.0)

In [10]:
loader = PyPDFLoader(file_path=input("Enter the pdf path: "))
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
document_chunks = text_splitter.split_documents(loader.load())
vector_db = FAISS.from_documents(document_chunks, embeddings_model)
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 5}
)

Enter the pdf path: https://drive.google.com/file/d/19py9BJ8iEQFbBaJCT747UEvU2O89O9oC/view?usp=sharing


PdfStreamError: Stream has ended unexpectedly

In [11]:
import re

def get_gdrive_direct_link(url):
    # Regular expression to extract the file ID from a Google Drive URL
    match = re.search(r'/file/d/([a-zA-Z0-9_-]+)', url)
    if match:
        file_id = match.group(1)
        return f"https://drive.google.com/uc?export=download&id={file_id}"
    return url

# Get user input and convert if it's a Google Drive link
pdf_url = input("Enter the pdf path: ").strip()
resolved_url = get_gdrive_direct_link(pdf_url)
print(f"Resolved PDF URL: {resolved_url}")

# Load and split documents
loader = PyPDFLoader(file_path=resolved_url)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
document_chunks = text_splitter.split_documents(loader.load())

# Index in vector database
vector_db = FAISS.from_documents(document_chunks, embeddings_model)
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 5}
)
print("Database and retriever successfully initialized!")

Enter the pdf path: https://drive.google.com/file/d/19py9BJ8iEQFbBaJCT747UEvU2O89O9oC/view?usp=sharing
Resolved PDF URL: https://drive.google.com/uc?export=download&id=19py9BJ8iEQFbBaJCT747UEvU2O89O9oC
Database and retriever successfully initialized!


In [ ]:
while True:
  user_query = input("Enter the query related to pdf: ")
  if user_query == "exit":
    print("Shutting down RAG Engine.")
    break
  retrieved_docs = retriever.invoke(user_query)
  context_block = "\n---\n".join([doc.page_content for doc in retrieved_docs])
  rag_prompt = ChatPromptTemplate.from_messages([
    ("system", (
      "You are an elite research analyst. Answer the question using ONLY "
      "the provided context snippets. If you do not possess the "
      "information, state that clearly.\n\n"
      "Retrieved PDF Context:\n{context}"
      )),
      ("human", "{question}")
    ])
  final_output = (rag_prompt | llm).invoke({
      "context": context_block,
      "question": user_query
      })
  print(f"RAG Generation Output:\n{final_output.content[0]["text"]}")

Enter the query related to pdf: What are the skills of the candidate? 
RAG Generation Output:
Based on the provided document, the candidate's skills are as follows:

*   **Programming Languages:** Python, SQL, Scala
*   **Data Engineering:** ETL/ELT Pipelines, Data Modeling, Data Warehousing, Batch & Stream Processing
*   **Big Data Technologies:** Apache Spark, Hadoop, Kafka, Hive
*   **Databases:** PostgreSQL, MySQL, MongoDB, Snowflake, BigQuery
*   **Cloud Platforms:** 
    *   AWS (S3, Redshift, Glue, Lambda, ECS)
    *   GCP (BigQuery, Dataflow)
*   **Workflow Orchestration:** Apache Airflow, Prefect
*   **MLOps & DevOps:** Docker, GitHub
*   **Other:** Tableau (listed under the GitHub/LinkedIn section)
